# Analisi Portafoglio con Gemini AI
Questo notebook permette di testare e visualizzare l'analisi del portafoglio passo dopo passo.

In [5]:
import os
import json
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Import custom modules
from scraper import scrape_getquin
from ai_engine import analyze_portfolio

# Load environment variables
if os.path.exists(".env"):
    load_dotenv(".env")
elif os.path.exists(".env.example"):
    load_dotenv(".env.example")
else:
    load_dotenv()

print("Ambiente configurato correttamente.")

ImportError: cannot import name 'genai' from 'google' (unknown location)

In [6]:
!pip install -r requirements.txt


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
selenium 4.35.0 requires typing_extensions~=4.14.0, but you have typing-extensions 4.15.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached pandas-2.2.1-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/29.4 MB ? eta -:--:--
   --- ------------------------------------ 2.9/29.4 MB 12.9 MB/s eta 0:00:03
   ------ --------------------------------- 4.5/29.4 MB 10.3 MB/s eta 0:00:03
   ---------- ----------------------------- 7.9/29.4 MB 11.9 MB/s eta 0:00:02
   -------------- ------------------------- 10.5/29.4 MB 11.9 MB/s eta 0:00:02
   ----------------- ---------------------- 13.1/29.4 MB 11.9 MB/s eta 0:00:02
   ---------------------- ----------------- 16.3/29.4 MB 12.2 MB/s eta 0:00:02
   ---------------------- ----------------- 16.8/29.4 MB 12.4 MB/s eta 0:00:02
   ----------------------- ---------------- 17.0/29.4 MB 9.6 MB/s eta 0:00:02
   ------------------------- -------------- 18.9/29.4 MB 9.9 MB/s eta 0:00:02
   ---------------

## 1. Ricerca Screenshot
Cerca l'immagine più recente nella cartella `input/`.

In [2]:
print("Ricerca screenshot del portafoglio nella cartella 'input'...")
scrape_result = scrape_getquin()

if scrape_result["status"] == "error":
    print(f"Errore:\n{scrape_result['message']}")
    image_path = None
else:
    image_path = scrape_result.get("image_path")
    print(f"Screenshot trovato: {image_path}")

Ricerca screenshot del portafoglio nella cartella 'input'...
Cerco lo screenshot più recente del portafoglio nella cartella 'input'...
Trovato screenshot più recente: input\posizioni_30-03-2026_15-31-44.png
Screenshot trovato: input\posizioni_30-03-2026_15-31-44.png


## 2. Analisi con Gemini AI
Invia l'immagine a Gemini per estrarre i dati e fare l'analisi del sentiment.

In [3]:
if image_path:
    # Leggi i dati iniziali del portafoglio dal file .env (se presenti)
    initial_value = os.getenv("INITIAL_PORTFOLIO_VALUE", "55000")
    start_date = os.getenv("PORTFOLIO_START_DATE", "2025-09-01")
    
    print("Analisi del portafoglio con Gemini in corso (potrebbe richiedere un minuto)...")
    try:
        analysis_json = analyze_portfolio(image_path, initial_value=initial_value, start_date=start_date)
        
        # Pulizia stringa JSON se Gemini ha aggiunto markdown
        if analysis_json.startswith("```json"):
            analysis_json = analysis_json[7:]
        if analysis_json.endswith("```"):
            analysis_json = analysis_json[:-3]
            
        data = json.loads(analysis_json.strip())
        print("Analisi completata con successo!")
    except Exception as e:
        print(f"Errore durante l'analisi AI o il parsing JSON:\n{str(e)}")
        data = None
else:
    print("Nessun'immagine trovata. Impossibile procedere con l'analisi.")
    data = None

Analisi del portafoglio con Gemini in corso (potrebbe richiedere un minuto)...
Errore durante l'analisi AI o il parsing JSON:
name 'analyze_portfolio' is not defined


## 3. Visualizzazione Risultati
Mostra il riassunto del portafoglio.

In [ ]:
if data:
    summary = data.get("portfolio_summary", {})
    df_summary = pd.DataFrame([summary])
    
    # Rinomina le colonne in UPPERCASE
    df_summary.columns = df_summary.columns.str.upper()
    
    if "STRATEGY_SUMMARY" in df_summary.columns:
        strategy = df_summary["STRATEGY_SUMMARY"].iloc[0]
        df_summary_display = df_summary.drop(columns=["STRATEGY_SUMMARY"])
    else:
        strategy = "N/A"
        df_summary_display = df_summary.copy()
        
    # Formattazione percentuale per il summary
    if "PERCENTAGE_RETURN" in df_summary_display.columns:
        df_summary_display["PERCENTAGE_RETURN"] = df_summary_display["PERCENTAGE_RETURN"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
        
    display(Markdown("### 📊 RIASSUNTO PORTAFOGLIO"))
    display(df_summary_display)
    display(Markdown(f"**💡 Strategia e Conclusioni:**\n{strategy}"))

Mostra l'analisi dettagliata degli asset e il sentiment.

In [ ]:
if data:
    assets = data.get("assets", [])
    df_assets = pd.DataFrame(assets)
    
    if not df_assets.empty:
        # Rinomina le colonne in UPPERCASE
        df_assets.columns = df_assets.columns.str.upper()
        
        df_assets_display = df_assets.copy()
        
        # Formattazione percentuale per gli asset
        if "WEIGHT_PERCENTAGE" in df_assets_display.columns:
            df_assets_display["WEIGHT_PERCENTAGE"] = df_assets_display["WEIGHT_PERCENTAGE"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
        if "PROFIT_LOSS_PERCENT" in df_assets_display.columns:
            df_assets_display["PROFIT_LOSS_PERCENT"] = df_assets_display["PROFIT_LOSS_PERCENT"].apply(lambda x: f"{x}%" if pd.notna(x) and str(x).strip() != "" else x)
            
        # Formattazione icone per il sentiment
        if "SENTIMENT" in df_assets_display.columns:
            def format_sentiment(s):
                if pd.isna(s): return s
                s_str = str(s).strip().title()
                if "Bullish" in s_str: return f"🟢 ↗️ {s_str}"
                if "Bearish" in s_str: return f"🔴 ↘️ {s_str}"
                if "Neutral" in s_str: return f"⚪ ＝ {s_str}"
                return s_str
            df_assets_display["SENTIMENT"] = df_assets_display["SENTIMENT"].apply(format_sentiment)
            
        # Mostriamo le colonne principali per la tabella
        display_cols = ["NAME", "TICKER", "POSITION_VALUE", "PROFIT_LOSS_EUR", "PROFIT_LOSS_PERCENT", "WEIGHT_PERCENTAGE", "SENTIMENT"]
        available_cols = [c for c in display_cols if c in df_assets_display.columns]
        
        display(Markdown("### 📈 ANALISI ASSET E SENTIMENT"))
        display(df_assets_display[available_cols])
        
        display(Markdown("### 📰 Dettaglio News Sentiment"))
        news_cols = ["NAME", "SENTIMENT", "NEWS_SUMMARY"]
        available_news_cols = [c for c in news_cols if c in df_assets_display.columns]
        
        display(df_assets_display[available_news_cols])
    else:
        print("Nessun asset trovato.")

## 4. Esportazione Dati
Salva i risultati in formato CSV nella cartella `output/`.

In [ ]:
if data:
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    summary_file = os.path.join(output_dir, f"portfolio_summary_{timestamp}.csv")
    assets_file = os.path.join(output_dir, f"portfolio_assets_{timestamp}.csv")
    
    try:
        df_summary.to_csv(summary_file, index=False, sep=";")
        df_assets.to_csv(assets_file, index=False, sep=";")
        print(f"✅ Dati salvati con successo in formato CSV in:")
        print(f"   - {summary_file}")
        print(f"   - {assets_file}")
    except Exception as e:
        print(f"Errore durante il salvataggio dei file CSV: {e}")